# Tesla and GameStop Stock & Revenue Analysis

This project compares historical stock prices and revenue trends for Tesla and GameStop using Python.

The goal is to practice a beginner-friendly financial data workflow:
- collecting stock data with `yfinance`
- scraping revenue tables from public web pages
- cleaning tabular data with `pandas`
- creating interactive charts with `Plotly`

> Note: This notebook uses public financial data and does not contain private or company-specific information.

## 1. Setup

First, install and import the libraries used in this project.

In [ ]:
# Install required packages if needed
# If they are already installed, this cell can be skipped.
!pip install -q yfinance beautifulsoup4 lxml plotly requests

In [ ]:
import pandas as pd
import requests
import yfinance as yf
import plotly.graph_objects as go
from bs4 import BeautifulSoup

## 2. Helper Functions

To avoid repeating code, I created small helper functions for downloading stock data, scraping revenue data, and creating charts.

In [ ]:
def get_stock_data(ticker: str) -> pd.DataFrame:
    """Download historical stock data for a ticker using yfinance."""
    stock = yf.Ticker(ticker)
    data = stock.history(period="max")
    data.reset_index(inplace=True)
    return data


def clean_revenue_value(value: str) -> float:
    """Convert revenue strings like '$31,536' into numeric values."""
    if pd.isna(value):
        return None
    value = str(value).replace("$", "").replace(",", "").strip()
    if value == "" or value == "nan":
        return None
    return float(value)


def get_revenue_data(url: str) -> pd.DataFrame:
    """Scrape annual revenue data from a public financial table."""
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    tables = soup.find_all("table")

    revenue_data = []
    for table in tables:
        rows = table.find_all("tr")
        for row in rows:
            cols = row.find_all("td")
            if len(cols) == 2:
                date = cols[0].text.strip()
                revenue = cols[1].text.strip()
                if revenue:
                    revenue_data.append({"Date": date, "Revenue": revenue})

    df = pd.DataFrame(revenue_data)
    df["Revenue"] = df["Revenue"].apply(clean_revenue_value)
    df.dropna(inplace=True)
    df["Date"] = pd.to_datetime(df["Date"])
    return df


def make_stock_revenue_graph(stock_data: pd.DataFrame, revenue_data: pd.DataFrame, title: str):
    """Create an interactive chart comparing stock closing price and revenue."""
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=stock_data["Date"],
            y=stock_data["Close"],
            mode="lines",
            name="Stock Close Price"
        )
    )

    fig.add_trace(
        go.Scatter(
            x=revenue_data["Date"],
            y=revenue_data["Revenue"],
            mode="lines+markers",
            name="Revenue"
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title="Date",
        yaxis_title="USD / Revenue",
        hovermode="x unified"
    )

    fig.show()

## 3. Tesla Data

In this section, I download Tesla stock data and scrape annual Tesla revenue data from a public financial table.

In [ ]:
tesla_stock = get_stock_data("TSLA")
tesla_stock.head()

In [ ]:
tesla_revenue_url = "https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue"
tesla_revenue = get_revenue_data(tesla_revenue_url)
tesla_revenue.head()

## 4. GameStop Data

In this section, I repeat the same process for GameStop.

In [ ]:
gamestop_stock = get_stock_data("GME")
gamestop_stock.head()

In [ ]:
gamestop_revenue_url = "https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue"
gamestop_revenue = get_revenue_data(gamestop_revenue_url)
gamestop_revenue.head()

## 5. Tesla Stock Price and Revenue Visualization

The chart below compares Tesla's historical stock closing price with its reported revenue trend.

In [ ]:
make_stock_revenue_graph(
    tesla_stock,
    tesla_revenue,
    "Tesla Stock Price and Revenue Trend"
)

## 6. GameStop Stock Price and Revenue Visualization

The chart below compares GameStop's historical stock closing price with its reported revenue trend.

In [ ]:
make_stock_revenue_graph(
    gamestop_stock,
    gamestop_revenue,
    "GameStop Stock Price and Revenue Trend"
)

## 7. Basic Comparison

This section creates a small summary table for both companies.

In [ ]:
summary = pd.DataFrame({
    "Company": ["Tesla", "GameStop"],
    "Stock Rows": [len(tesla_stock), len(gamestop_stock)],
    "Revenue Rows": [len(tesla_revenue), len(gamestop_revenue)],
    "Latest Stock Close": [tesla_stock["Close"].iloc[-1], gamestop_stock["Close"].iloc[-1]],
    "Latest Revenue": [tesla_revenue["Revenue"].iloc[0], gamestop_revenue["Revenue"].iloc[0]]
})

summary

## Conclusion

This project demonstrates a simple financial data analysis workflow using Python. Tesla and GameStop are useful examples because their stock price behavior and business revenue patterns are very different.

Key skills practiced in this notebook:
- downloading stock data from an API-like Python library
- collecting revenue data through web scraping
- cleaning currency values for analysis
- creating interactive visualizations
- organizing a notebook as a portfolio project

This is a beginner-friendly project and can be improved further by adding more companies, calculating growth rates, or comparing stock price changes over selected time periods.